# Python to Fortran Code Converter

In [ ]:
# imports
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

from system_info import retrieve_system_info, fortran_toolchain_info


In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key: print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else: print("OpenRouter API Key not set (and this is optional)")

openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
clients = {
    "google/gemini-3.1-flash-lite-preview": openrouter,
    "x-ai/grok-4.1-fast": openrouter,
    "deepseek/deepseek-r1-0528": openrouter,
    "xiaomi/mimo-v2-flash": openrouter,
}

models = [ model for model in clients.keys() ]

In [ ]:
system_info = retrieve_system_info()
fortran_info = fortran_toolchain_info()
fortran_info 

In [ ]:
message = f"""
Here is a report of the system information for my computer.
I want to compile a single Fortran file called main.f90 and then execute it in the simplest way possible.
Please reply with whether I need to install a Fortran compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Fortran code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Fortran toolchain information:
{fortran_info}
"""

response = openrouter.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

In [ ]:
# hard code the compile command for fortran based on the above response and the system information

compile_command = [
    "/home/linuxbrew/.linuxbrew/bin/gfortran",
    "main.f90",
    "-o", "main",
    "-Ofast",
    "-march=native",
    "-g0",
    "-mtune=native",
    "-fopenmp",
    "-fomit-frame-pointer",
]

run_command = ["./main"]

## Let's generate Fortran code!

In [ ]:
language = "Fortran"
extension = "f90"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
The Fortran toolchain information is:
{fortran_info}
Your response will be written to a file called main.{extension} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [ ]:
fix_compile_system_prompt = """
You are a Fortran compiler error fixer. Your sole task is to fix every compiler error
in the provided Fortran code and return compilable Fortran source with zero prose.

Rules:
1. Output ONLY valid Fortran source code. No markdown, no fences, no explanations.
2. Fix every error listed in the compiler output — nothing more, nothing less.
3. Do not add new functionality. Do not optimise. Do not restructure working code.
4. Preserve all existing logic, comments, and formatting where possible.
"""

def fix_compile_issues_for(fortran_code, compiler_errors):
    return f"""
The Fortran code below was generated by an LLM and may contain syntax errors or
non-code text that prevents compilation. Apply every rule from your system prompt and
return only the corrected Fortran source — nothing else.

The code was run through gfortran -c (compile-only) and produced these errors:
--- COMPILER OUTPUT ---
{compiler_errors.strip()}
--- END COMPILER OUTPUT ---
Fix every error listed above in addition to applying all rules from your system prompt.

Fortran code to clean:

{fortran_code}
"""

In [ ]:

cleanup_system_prompt = f"""
You are a Fortran code validator and cleaner. Your sole task is to return clean, 
compilable Fortran 90/95 source code with zero prose.

Rules — follow ALL of them without exception:
1. Output ONLY valid Fortran source code. No markdown, no fences, no explanations.
2. Remove every line that is not Fortran code or a Fortran inline comment (!).
3. Remove all markdown code fences (```fortran, ```f90, ```).
4. Fix ALL of the following syntax errors if present:
   a. External functions called from `program` without an interface or `external` declaration — 
add `<return_type>, external :: <name>` in the declarations section, OR restructure using `contains`.
   b. Variables used but never declared — implicit none is assumed, so declare everything.
   c. Integer literals that overflow default INTEGER — use integer(8) or the _8 suffix.
   d. Mismatched `end` statements (e.g. bare `end` instead of `end program main`).
   e. Wrong format descriptors in `print`/`write` that would cause a runtime format error.
   f. Missing `intent(in/out/inout)` on dummy arguments — add them.
   g. `result` variable used before assignment in a function.
   h. Mixed default/explicit integer kinds in arithmetic — add int(...,8) or int(...,kind=4) casts.
   i. Any construct not supported by gfortran 15 / Fortran 95 — replace with standard equivalents.
5. Do not add new functionality. Do not optimise further. Fix only what is broken.
6. Preserve all existing ! comments. Remove any prose masquerading as comments.
7. The output must compile cleanly with:
   gfortran main.f90 -o main -O3 -march=native -ffast-math -g0
8. Timing code — enforce the following standard pattern wherever the code measures elapsed time:
   a. Use `system_clock` for wall-clock time (matches Python's `time.time()`), NOT `cpu_time`.
      Correct pattern:
        integer(8) :: t_start, t_end, t_rate
        call system_clock(t_start, t_rate)
        ! ... timed work ...
        call system_clock(t_end)
        print '(A, F0.6, A)', "Execution Time: ", real(t_end - t_start, 8) / real(t_rate, 8), " seconds"
   b. If the code uses `cpu_time` instead, replace it with the `system_clock` pattern above.
   c. Declare timing variables as `integer(8)` (not default integer) to avoid counter overflow
      on long-running programs.
   d. The elapsed-time print line must use the format `"Execution Time: <value> seconds"` with
      6 decimal places of precision, matching the Python output format exactly.
   e. Place the `call system_clock(t_start, t_rate)` immediately before the timed work and
      `call system_clock(t_end)` immediately after — no extra statements between them and the
      timed call.
"""


def cleanup_prompt_for(fortran_code):
    return f"""
The Fortran code below was generated by an LLM and may contain syntax errors or
non-code text that prevents compilation. Apply every rule from your system prompt and
return only the corrected Fortran source — nothing else.

The code compiled without errors in the compile-only check.
Still apply all rules from your system prompt (remove prose, fix timing, etc.).

Fortran code to clean:

{fortran_code}
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [ ]:
def port(model, python):
    client = clients[model]
    MAX_FIX_ITERATIONS = 1

    # Build compile-check command: same flags as compile_command but -c (no link, no run)
    compile_check_cmd = (
        [compile_command[0], f"main.{extension}", "-c", "-o", "main.o"]
        + [flag for flag in compile_command if flag.startswith("-") and flag != "-o"]
    )

    chat_history = []

    # First call: generate Fortran from Python
    first_messages = messages_for(python)
    response1 = client.chat.completions.create(
        model=model,
        messages=first_messages,
    )
    first_reply = response1.choices[0].message.content
    current_code = first_reply.replace('```fortran','').replace('```f90','').replace('```','')
    chat_history.append({"role": "user",      "content": user_prompt_for(python)})
    chat_history.append({"role": "assistant", "content": first_reply})

    # Fix compile errors loop
    for _ in range(MAX_FIX_ITERATIONS):
        write_output(current_code)
        check_result = subprocess.run(compile_check_cmd, text=True, capture_output=True)
        compiler_errors = check_result.stderr
        if check_result.returncode == 0:
            break
        fix_messages = [
            {"role": "system",    "content": fix_compile_system_prompt},
            first_messages[1],                                              # initial user prompt
            {"role": "assistant", "content": current_code},
            {"role": "user",      "content": fix_compile_issues_for(current_code, compiler_errors)},
        ]
        response = client.chat.completions.create(
            model=model,
            messages=fix_messages,
        )
        code_before = current_code
        current_code = (
            response.choices[0].message.content
            .replace('```fortran','').replace('```f90','').replace('```','')
        )
        chat_history.append({"role": "user",      "content": fix_compile_issues_for(code_before, compiler_errors)})
        chat_history.append({"role": "assistant", "content": current_code})

    # Single cleanup pass after loop
    cleanup_messages = [
        {"role": "system",    "content": cleanup_system_prompt},
        first_messages[1],                                                  # initial user prompt
        {"role": "assistant", "content": current_code},
        {"role": "user",      "content": cleanup_prompt_for(current_code)},
    ]
    code_before = current_code
    response = client.chat.completions.create(
        model=model,
        messages=cleanup_messages,
    )
    current_code = (
        response.choices[0].message.content
        .replace('```fortran','').replace('```f90','').replace('```','')
    )
    chat_history.append({"role": "user",      "content": cleanup_prompt_for(code_before)})
    chat_history.append({"role": "assistant", "content": current_code})

    return current_code, chat_history

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# Use the gfortran compile_command and run_command defined above

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
import re

def extract_time(output):
    match = re.search(r'Execution Time:\s*([\d.]+)\s*seconds', output)
    return float(match.group(1)) if match else None

def convert(model, python):
    fortran_code, chat_history = port(model, python)
    python_output  = run_python(python)
    fortran_output = compile_and_run(fortran_code)

    py_time = extract_time(python_output)
    ft_time = extract_time(fortran_output)

    if py_time and ft_time and ft_time > 0:
        speedup = f"\n\nSpeedup: {py_time / ft_time:,.1f}x faster than Python"
    else:
        speedup = ""

    combined = (
        f"=== Python Output ===\n{python_output.strip()}"
        f"\n\n=== Fortran Output ===\n{fortran_output.strip()}"
        f"{speedup}"
    )
    return fortran_code, combined, chat_history

In [ ]:
python_hard = """# Be careful to support large numbers

# Linear Congruential Generator (LCG) implementation and maximum subarray sum calculation

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 100000        # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=10
            )
        with gr.Column(scale=6):
            fortran = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=10
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], show_label=False)
        run_btn = gr.Button("Convert", elem_classes=["convert-btn"])

    with gr.Row():
        fortran_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    with gr.Row():
        history_display = gr.Chatbot(label="LLM Call History", height=400, type="messages")

    run_btn.click(fn=convert, inputs=[model, python], outputs=[fortran, fortran_out, history_display])

ui.launch(inbrowser=True)